# Pilot 3 · validate and measure the activation instrument

**Question:** How much do the saved rank-1 SFT adapters change activations, and where along the 28 decoder blocks do those changes appear?

Pilot 1 established evaluation and scoring. Pilot 2 trained and evaluated checkpoints **0, 8, 16, 32, 64**. Pilot 3 reuses those adapters and the **same 150 held-out GSM8K items**, adding 150 pinned FineWeb passages as a descriptive control. It does **no training, answer generation or answer scoring**.

We feed identical predetermined tokens to the untuned and adapted model. GSM8K includes complete questions and gold solutions; FineWeb supplies 128 content tokens per selected passage. Question, solution and passage masks keep chat scaffolding out of aggregates while retaining it as model context.

**Two instruments:** block-output differences include upstream changes; direct module contributions measure each projection's LoRA product on its adapted-pass input. Neither is a weight-norm ratio or a causal attribution. The primary block ratio is `norm(mean change) / mean baseline norm`; the module ratio is `mean direct norm / mean ordinary-output norm`. Compare token weighting with equal-example weighting.

**Flow:** setup → verify sources → freeze/audit tokens → validate identities → profile/review batches → freeze execution → measure/resume → verify → CPU report.

Local CPU tests establish software behavior with fixtures and tiny locally constructed models. **Actual saved-checkpoint checks, CUDA capacity and numerical agreement must pass here in Colab.** Earlier generation speed estimates do not predict this forward-only workload. Geometry is descriptive, **not causal** evidence of capability loss or a recommendation for layer placement.

[Interview and decisions](https://github.com/mahadikprasad15/spar-pilot/blob/main/.scratch/pilot3-write/interview.md) · [Spec](https://github.com/mahadikprasad15/spar-pilot/blob/main/.scratch/pilot3-write/spec.md) · [Technical design](https://github.com/mahadikprasad15/spar-pilot/blob/main/.scratch/pilot3-write/design.md) · [Earlier-pilot lessons](https://github.com/mahadikprasad15/spar-pilot/blob/main/.scratch/pilot3-write/prior-pilot-learnings.md) · [Instrument guide](https://github.com/mahadikprasad15/spar-pilot/blob/main/docs/pilot3-instrument.md)

Do not use **Run all**: the environment restart and the two manual reviews are deliberate stopping points.

## 1. Mount Drive and restore the pinned code

Choose one GPU for sections 3–11; there is **no GPU-family whitelist**. Capacity and speed are measured before production. FP32 is fixed, with no quantization, autocast or TF32.

The code checkout `/content/spar-pilot` is temporary; results in `SPAR/spar-pilot/artifacts` are persistent. **Both pilots' earlier artifacts stay in that shared tree.** Do not move adapter files or edit saved configs. Set `SOURCE_PLAN` to your completed Pilot 2 plan; candidate paths are printed below. Names for preparation, production and reporting are distinct and immutable.

`HARNESS_COMMIT` pins the implementation already containing all Pilot 3 commands. Reconnection fetches that exact revision. If Git cannot find it, confirm it has been pushed to the remote. Local edits block checkout so they can be preserved, rather than discarded.

**CPU report later:** set `CPU_REPORT_ONLY=True`, then run 1, 2 if needed/restart, 3, 4, 11, 12, 13. Keep all names and the artifact root identical; do not repeat GPU stages.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys, time
from google.colab import drive

drive.mount('/content/drive')
REPO_URL = 'https://github.com/mahadikprasad15/spar-pilot.git'
HARNESS_COMMIT = '9e5d56ea96d111b1eb66ecaa783d19743afd1d3d'
REPO_DIR = Path('/content/spar-pilot')
ARTIFACT_ROOT = Path('/content/drive/MyDrive/SPAR/spar-pilot/artifacts')
CPU_REPORT_ONLY = False # @param {type:"boolean"}
SOURCE_PLAN = 'pilot2-sft-fp32-l4-batch8-v1' # @param {type:"string"}
PREPARATION_NAME = 'pilot3-inputs-v1' # @param {type:"string"}
RUN_NAME = 'pilot3-write-v2' # @param {type:"string"}
REPORT_NAME = 'pilot3-write-report-v2' # @param {type:"string"}
assert PREPARATION_NAME != RUN_NAME, 'Preparation and execution need distinct names.'
SOURCE_CONFIG = ARTIFACT_ROOT / 'plans' / SOURCE_PLAN / 'sft.config.json'
PREPARED_PATH = ARTIFACT_ROOT / 'plans' / PREPARATION_NAME / 'activation.prepared.json'
EXECUTION_PATH = ARTIFACT_ROOT / 'plans' / RUN_NAME / 'activation.execution.json'
REPORT_DIR = ARTIFACT_ROOT / 'reports' / REPORT_NAME
SESSION_DIR = ARTIFACT_ROOT / 'notebook' / RUN_NAME

boot = Path('/proc/sys/kernel/random/boot_id').read_text().strip()
for process_path in SESSION_DIR.glob('*.process.json'):
    record = json.loads(process_path.read_text())
    if record.get('running') and record.get('boot_id') == boot:
        try:
            os.kill(record['pid'], 0)
        except ProcessLookupError:
            pass
        else:
            raise RuntimeError(f"Inspect active child PID {record['pid']} before switching code.")

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
dirty = subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO_DIR, text=True).strip()
assert not dirty, 'Local edits detected. Inspect git status; preserve them before switching code.'
subprocess.run(['git', 'fetch', 'origin', HARNESS_COMMIT], cwd=REPO_DIR, check=True)
subprocess.run(['git', 'checkout', '--detach', HARNESS_COMMIT], cwd=REPO_DIR, check=True)
actual = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
assert actual == HARNESS_COMMIT
assert all((REPO_DIR / name).is_file() for name in [
    'pilot_eval/activation_prepare.py', 'pilot_eval/activation_workflow.py',
    'pilot_eval/activation_profile.py', 'pilot_eval/activation_measurement.py',
    'pilot_eval/activation_report.py', 'requirements-pilot2.txt'])
SESSION_DIR.mkdir(parents=True, exist_ok=True)
print('Pinned checkout:', actual)
print('Persistent root:', ARTIFACT_ROOT)
print('Selected Pilot 2 source:', SOURCE_CONFIG, '| exists:', SOURCE_CONFIG.is_file())
print('Available Pilot 2 configs:')
for candidate in sorted((ARTIFACT_ROOT / 'plans').glob('*/sft.config.json')):
    print(' ', candidate.relative_to(ARTIFACT_ROOT))

## 2. Install dependencies, then restart the session

For GPU work, reuse the exact library pins validated by Pilot 2 and required by the activation backend; its filename is `requirements-pilot2.txt`. No SFT trainer is invoked in this pilot. CPU reporting needs only the package's NumPy/Matplotlib dependencies.

**After installing, use Runtime → Restart session**, rerun section 1, skip this install cell, then continue at 3. Installing into an already imported environment does not refresh modules in memory. The CLI subprocesses below use fresh imports.

Installation downloads packages. Preparation may download dataset/tokenizer files; instrument validation loads pinned Qwen weights. Existing caches are reused.

In [ ]:
if CPU_REPORT_ONLY:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(REPO_DIR) + '[report]'], check=True)
else:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO_DIR / 'requirements-pilot2.txt')], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '-e', str(REPO_DIR)], check=True)
print('Restart session, rerun section 1, skip section 2, then continue at section 3.')

## 3. Verify entry points, versions and local CPU tests

All eight activation commands must be visible before expensive work. GPU mode requires pinned packages and one CUDA device; **any device passing the instrument and memory checks is eligible**. Torch runs only in a child, keeping the notebook kernel free of model/GPU ownership.

The offline CPU suite downloads no models. It includes tiny locally constructed model tests and saved-workflow fixtures. Expected: tests pass. This is a software gate; real-source/CUDA validation follows in section 7. CPU-report mode verifies commands and reporting packages without importing GPU libraries.
Calculation checks compare summaries against independent reductions of the same forward-pass outputs, for single examples and padded batches. Numerical agreement across batch shapes is checked separately on the actual model in section 8; its thresholds and stop-on-mismatch rule are unchanged.


In [ ]:
import importlib.metadata
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
sys.path.insert(0, str(REPO_DIR))
help_result = subprocess.run([sys.executable, '-m', 'pilot_eval', '--help'], cwd=REPO_DIR,
                             capture_output=True, text=True, check=True)
commands = ['activation-prepare', 'activation-audit', 'activation-validate', 'activation-profile',
            'activation-calibration-prepare', 'activation-calibrate', 'activation-calibration-freeze', 'activation-calibration-validate', 'activation-freeze', 'activation-measure', 'activation-verify', 'activation-report']
assert all(command in help_result.stdout for command in commands), 'Missing entry points: wrong checkout.'
for package in ['numpy', 'matplotlib']:
    print(package, importlib.metadata.version(package))
if not CPU_REPORT_ONLY:
    from pilot_eval.sft_backend import PINS
    for package, expected in PINS.items():
        actual = importlib.metadata.version(package)
        assert actual.split('+')[0] == expected, f'{package}: {actual}; expected {expected}. Install/restart.'
        print(package, actual)
    subprocess.run([sys.executable, '-c',
        "import torch; assert torch.cuda.is_available() and torch.cuda.device_count() == 1; print('GPU:', torch.cuda.get_device_name(0))"], check=True)
    subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=REPO_DIR, check=True,
                   env=dict(os.environ, HF_HUB_OFFLINE='1', HF_DATASETS_OFFLINE='1'))
else:
    print('CPU report mode: no CUDA check or model-library imports required.')

## 4. Restore state and command helpers

Every scientific stage calls the package CLI; this notebook does not implement activation math. Logs and child identities are saved to Drive. Status reads tolerate missing/empty/partial JSON; a command's exit code is checked separately. **Only `activation-verify` establishes scientific completion.**

If a cell is interrupted, its child may continue. The helper leaves its PID record available and blocks starting another live child. After a runtime replacement, a different Linux boot identity distinguishes obsolete PID records. Inspect section 14 before rerunning an interrupted stage. Completed scientific shards are resumed by the package, not inferred from notebook variables.

On a reconnect rerun 1, 3, 4; saved preparation and execution are restored from their immutable files.

In [ ]:
from pilot_eval.activation_prepare import load_prepared
from pilot_eval.activation_measurement import load_execution, monitor_activation_process

def restore_preparation():
    return load_prepared(PREPARED_PATH, ARTIFACT_ROOT)

def restore_execution():
    return load_execution(EXECUTION_PATH, ARTIFACT_ROOT)

def boot_identity():
    return Path('/proc/sys/kernel/random/boot_id').read_text().strip()

def live_children():
    boot = boot_identity()
    children = []
    for path in SESSION_DIR.glob('*.process.json'):
        try:
            record = json.loads(path.read_text())
            if record.get('running') and record.get('boot_id') == boot:
                os.kill(record['pid'], 0)
                proc_stat = Path(f"/proc/{record['pid']}/stat")
                state = proc_stat.read_text().rsplit(')', 1)[1].split()[0] if proc_stat.exists() else None
                if state != 'Z':
                    children.append(record)
        except ProcessLookupError:
            pass
        except (OSError, ValueError, KeyError) as error:
            raise RuntimeError(f'Cannot safely inspect process record {path}: {error}') from error
    return children

def run_command(arguments, label, run_dir=None):
    global ACTIVE_PROCESS
    active = live_children()
    if active:
        raise RuntimeError(f'An earlier child may still be alive: {active}. Inspect section 14 first.')
    command = [sys.executable, '-u', '-m', 'pilot_eval', *map(str, arguments),
               '--output-root', str(ARTIFACT_ROOT)]
    console_path = SESSION_DIR / (label + '.console.log')
    process_path = SESSION_DIR / (label + '.process.json')
    boot = boot_identity()
    print('Starting:', label, flush=True)
    with console_path.open('a') as console:
        process = subprocess.Popen(command, cwd=REPO_DIR, stdout=console, stderr=subprocess.STDOUT)
        ACTIVE_PROCESS = process
        record = dict(pid=process.pid, command=command, running=True, boot_id=boot)
        process_path.write_text(json.dumps(record))
        console_start = console.tell()
        last_snapshot, last_message, last_print = None, None, 0.
        while True:
            observation = monitor_activation_process(process, run_dir or SESSION_DIR, last_snapshot=last_snapshot)
            last_snapshot = observation['snapshot']
            snapshot = observation['snapshot']
            message = (f"{label}: {snapshot['state']} {snapshot['completed']}/{snapshot['total']}"
                       if snapshot else f'{label}: process {process.pid} active')
            # Read only this attempt's recent output, without rereading the whole log.
            with console_path.open('rb') as recent:
                recent.seek(0, 2)
                recent.seek(max(console_start, recent.tell() - 8192))
                stages = [line for line in recent.read().decode(errors='replace').splitlines()
                          if line.startswith(('activation profile:', 'activation instrument:', 'activation calibration:'))]
            if stages:
                message += ' | ' + stages[-1]
            if snapshot and not observation['snapshot_is_current']:
                message += ' (last known status; current read unavailable)'
            if message != last_message or time.monotonic() - last_print >= 60:
                print(message, flush=True)
                last_message, last_print = message, time.monotonic()
            if process.poll() is not None:
                break
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                pass
    record.update(running=False, returncode=process.returncode)
    process_path.write_text(json.dumps(record))
    if process.returncode:
        print('\n'.join(console_path.read_text().splitlines()[-30:]))
        raise RuntimeError(f'Command failed; inspect {console_path} and run logs. No settings were changed.')
    print('Command exited successfully:', label, '| console:', console_path)
def calibrated_profile_directory():
    from pilot_eval.workflow import _hash
    prepared, _ = restore_preparation()
    plan = json.loads(CALIBRATION_PATH.read_text())
    rule = json.loads((ARTIFACT_ROOT / plan['run_path'] / 'rule.json').read_text())
    return ARTIFACT_ROOT / prepared['run_path'] / ('profile-v2-' + _hash(rule)[:12])


## 5. Verify Pilot 2 sources and freeze the measurement inputs

Preparation verifies the complete training evidence and actual adapter files at all five steps, not just their folder names. It copies no adapters and trains nothing. Expected: **300 fixed rows** — 150 original GSM8K test items and 150 FineWeb passages — plus hashes, tokenizer offsets, masks and an audit.

GSM8K solutions retain calculator marks and final-answer markers, with no truncation. FineWeb uses pinned `sample-10BT`: first 2,000 streamed documents, eligible documents sampled with seed 42, first 128 content tokens retained. This is a bounded convenience control, not a representative web sample.

Rerunning the same preparation verifies saved artifacts rather than drawing another cohort. Missing or corrupt source evidence stops. Set the source path in section 1 correctly; do not create replacement source markers.

In [ ]:
assert not CPU_REPORT_ONLY, 'Skip preparation in CPU report mode.'
assert SOURCE_CONFIG.is_file(), f'Choose the actual completed Pilot 2 config: {SOURCE_CONFIG}'
run_command(['activation-prepare', '--source-config', SOURCE_CONFIG,
             '--name', PREPARATION_NAME, '--fineweb-config', 'sample-10BT'], 'prepare')
PREPARED, ROWS = restore_preparation()
print('Frozen examples:', len(ROWS))
print('Prepared config:', PREPARED_PATH)
print('Input evidence:', ARTIFACT_ROOT / PREPARED['items_path'])
print('FineWeb:', json.dumps(PREPARED['fineweb'], indent=2))
print('Actual source checkpoints:')
for step, evidence in PREPARED['source_evidence']['checkpoints'].items():
    print(step, ARTIFACT_ROOT / evidence['path'], '| adapter sha256:', evidence['adapter_sha256'])

## 6. Inspect the rendered inputs and token masks

The audit reports lengths, exclusions and selected identities. Inspect one GSM8K question/solution and one FineWeb passage. A mask selects tokens for measurement; it does not remove other tokens from the context. System text, task instructions, chat controls, padding and ambiguous boundary tokens are excluded from aggregates.

Review that complete gold solutions and all three views are present. Longer solutions receive more weight under token weighting; equal-example weighting tests sensitivity to this choice. **These are gold input lengths, not generated response lengths.**

Set the review checkbox and notes only after reading the evidence. Notes are carried into the frozen execution review. Reruns simply redisplay verified inputs.

In [ ]:
from IPython.display import Markdown, display
PREPARED, ROWS = restore_preparation()
run_command(['activation-audit', '--config', PREPARED_PATH], 'audit')
display(Markdown((ARTIFACT_ROOT / PREPARED['audit_path']).read_text()))
for corpus in ['gsm8k', 'fineweb']:
    row = next(row for row in ROWS if row['corpus'] == corpus)
    print('\nID:', row['id'], '| corpus:', corpus)
    print(row['rendered'])
    print('Counted tokens:', row['counts'])
    print('Ambiguous positions:', row['ambiguous_positions'])
    for view in ['question', 'solution', 'user']:
        positions = [i for i, chosen in enumerate(row['masks'][view]) if chosen]
        print(view, '| first token positions:', positions[:12], '| total:', len(positions))
INPUTS_REVIEWED = False # @param {type:"boolean"}
INPUT_REVIEW_NOTES = '' # @param {type:"string"}

## 7. Validate the real instrument on one diagnostic batch

This loads the actual pinned FP32 model and saved adapters. On the longest GSM8K sequence plus the longest FineWeb passage it checks **exactly zero step-0 writes**, unchanged frozen weights, identical disabled-adapter references across checkpoint switches and rank-1 identities in every adapted projection.

The rank-1 identity is checked before addition and after adapted-minus-ordinary subtraction, on up to 16 fixed counted positions per module/view/checkpoint. The scientific summaries count every selected token. Fixed thresholds are engineering acceptance rules, not universal error bounds. Hooks are decoder-block outputs before the final model normalization.

Expected: all gates pass, saved validation diagnostics. **This is diagnostic-only, not a full-cohort result.** OOM or identity errors stop; preserve diagnostics and inspect rather than altering thresholds. Main batches remain to be profiled next.

In [ ]:
assert not CPU_REPORT_ONLY
assert INPUTS_REVIEWED and INPUT_REVIEW_NOTES.strip(), 'Review section 6 inputs and add notes first.'
PREPARED, _ = restore_preparation()
run_command(['activation-validate', '--config', PREPARED_PATH, '--batch-size', '2', '--namespace', 'agreement-v2'], 'validate')
print('Diagnostic evidence:', ARTIFACT_ROOT / PREPARED['run_path'] / 'validation-v2')

# 8a. Measure the numerical noise envelope

This uses 16 new frozen examples: eight GSM8K and eight FineWeb, excluding the old stress profile. It collects batch 1, a repeat, deliberate right padding, and batches 2/4/8/16 across all five adapters. One model is reused within each phase. Each completed variant is saved and verified on resume; failures do not change thresholds. This requires the GPU and can take several minutes per variant. Live output shows the current variant, example offset and checkpoint.


In [ ]:
assert not CPU_REPORT_ONLY
assert INPUTS_REVIEWED and INPUT_REVIEW_NOTES.strip(), 'Review prepared inputs first.'
CALIBRATION_NAME = 'pilot3-noise-v2' # @param {type:"string"}
CALIBRATION_PATH = ARTIFACT_ROOT / 'plans' / CALIBRATION_NAME / 'activation.calibration.json'
run_command(['activation-calibration-prepare', '--config', PREPARED_PATH, '--name', CALIBRATION_NAME], 'calibration-prepare')
CALIBRATION_PLAN = json.loads(CALIBRATION_PATH.read_text())
CALIBRATION_DIR = ARTIFACT_ROOT / CALIBRATION_PLAN['run_path']
run_command(['activation-calibrate', '--config', CALIBRATION_PATH], 'calibrate', CALIBRATION_DIR / 'calibration')


# 8b. Review and freeze the rule

Review the envelope and unresolved-direction counts below. Confirm cohort identity, repeat and padding controls, finite error scales, and any OOM exclusions. The approved rule uses three times the measured maximum error, not a confidence interval. Vectors smaller than the measured resolution are flagged; their directions should not be treated as reliable. Add notes before freezing. Validation data has not been collected yet.


In [ ]:
from pilot_eval.activation_calibration import comparison_metrics, fit_rule
from pilot_eval.activation_profile import _load_arrays
CALIBRATION_PLAN = json.loads(CALIBRATION_PATH.read_text())
CALIBRATION_DIR = ARTIFACT_ROOT / CALIBRATION_PLAN['run_path']
evidence = json.loads((CALIBRATION_DIR / 'calibration/results.json').read_text())
def calibration_arrays(variant):
    return {step: _load_arrays(CALIBRATION_DIR / 'calibration' / variant / f'step-{step}.npz') for step in (0,8,16,32,64)}
reference = calibration_arrays('batch-1')
metrics = [comparison_metrics(reference, calibration_arrays(r['variant'])) for r in evidence['records'] if r['status']=='completed' and r['variant']!='batch-1']
print('Variant status:', [(r['variant'],r['status']) for r in evidence['records']])
print('Provisional envelope:', json.dumps(fit_rule(metrics, margin=3), indent=2))
print('Coverage:', [m['coverage'] for m in metrics])
CALIBRATION_REVIEWED = False # @param {type:"boolean"}
CALIBRATION_NOTES = '' # @param {type:"string"}
if CALIBRATION_REVIEWED:
    assert CALIBRATION_NOTES.strip(), 'Add calibration review notes.'
    run_command(['activation-calibration-freeze', '--config', CALIBRATION_PATH, '--review-notes', CALIBRATION_NOTES], 'calibration-freeze')
else:
    print('Review evidence, check the box, add notes and rerun this cell.')


# 8c. Validate on 16 separate examples

Only after freezing the rule, collect the independent validation cohort. Repeatability and padding must pass. Wrong-sign, doubled-scale and corrupted-count controls must be rejected. Batch candidates exceeding the frozen envelope are excluded. No threshold is adjusted using validation outcomes. The subsequent stress profile remains an additional gate.


In [ ]:
assert not CPU_REPORT_ONLY
assert CALIBRATION_REVIEWED and CALIBRATION_NOTES.strip(), 'Freeze the reviewed rule first.'
run_command(['activation-calibration-validate', '--config', CALIBRATION_PATH], 'calibration-validate', CALIBRATION_DIR / 'validation')
validated = json.loads((CALIBRATION_DIR / 'validated/results.json').read_text())
print('Independently validated batches:', validated['validated_batches'])
print('Rejected negative controls:', validated['negative_controls'])
print('Validation disagreements/coverage:', json.dumps(validated['agreements'], indent=2))


## 8. Profile forward-pass batches 1, 2, 4, 8, 16

The fixed workload contains eight longest GSM8K sequences and eight longest passages. Each candidate runs the real reference cache plus all five adapters, with warmup, synchronized timing and memory measurement. It compares saved summaries and coverage with batch 1 using the frozen agreement thresholds.

An OOM marks that candidate unsuitable and profiling continues; numerical/identity errors stop. Timed scope excludes loading, warmup and disk writes. Profiling checks overhead separately from main adapted passes where available. No generation tokens/sec estimate is reused.

Expected: a saved candidate table and a provisional fastest passing batch. Reruns verify cached profiling; a changed runtime/code identity requires preserving prior diagnostic/profile evidence and revalidating before a new execution freeze.
Profiling verifies exact frozen-base SHA256 before and after each warmup/timed workload, with parameter identity/version guards between measurements. Candidate files are written only after the final hash passes. Stage progress appears in the cell and console log.


In [ ]:
assert not CPU_REPORT_ONLY
PREPARED, _ = restore_preparation()
PROFILE_DIR = calibrated_profile_directory()
run_command(['activation-profile', '--config', PREPARED_PATH, '--calibration', CALIBRATION_PATH], 'profile-v2', PROFILE_DIR)


## 9. Review profiling evidence and select a passing batch

This displays persisted timings, memory and agreement. Input tokens/sec measures completion of the whole profiled workload, not autoregressive throughput. Capacity must be checked on these actual longest inputs; do not infer it from the earlier GSM8K generation benchmark.

`0` below means propose the fastest passing candidate; you may choose another passing size explicitly. Read the saved candidate evidence, check the review box and add notes before freezing. The package verifies the selected profile and diagnostic markers again. Review does not relax a failed gate.

In [ ]:
PREPARED, _ = restore_preparation()
if 'PROFILE_DIR' not in globals():
    PROFILE_DIR = calibrated_profile_directory()
PROFILE = json.loads((PROFILE_DIR / 'results.json').read_text())
lines = ['| Batch | Status | Seconds | Input tokens/sec | Peak allocated GiB | Agreement |',
         '|---:|---|---:|---:|---:|---|']
for row in PROFILE['measurements']:
    if row['status'] == 'passed':
        lines.append(f"| {row['batch_size']} | passed | {row['seconds']:.2f} | {row['input_tokens_per_second']:.2f} | {row['memory']['allocated_bytes']/2**30:.2f} | {row['agreement']['passed']} |")
    else:
        lines.append(f"| {row['batch_size']} | {row['status']} | — | — | — | — |")
display(Markdown('\n'.join(lines)))
print('Provisional fastest:', PROFILE['provisional_fastest_batch'])
print('All evidence:', PROFILE_DIR)
SELECTED_BATCH_SIZE = 0 # @param [0, 1, 2, 4, 8, 16] {type:"raw"}
BENCHMARK_REVIEWED = False # @param {type:"boolean"}
BENCHMARK_NOTES = '' # @param {type:"string"}

## 10. Freeze the reviewed execution and measure/resume

Freezing saves the exact ordered batch memberships, masks/input identities, precision, runtime and validation policy. Production is **one untuned block reference plus five adapted main passes per batch**, with additional validation overhead. One model is loaded; examples within a batch run together, while adapters run sequentially.

Each batch saves one baseline shard and five checkpoint-specific shards, with hashes and completion markers written last. Per-example sufficient summaries support later weighting/bootstrap calculations; full token activations are not retained. Temporary reference activations are released after each batch.

Rerunning skips verified completed combinations. Partial batches recreate and verify the baseline, then run only missing checkpoints. Production OOM stops with saved progress; it does **not** automatically shrink batches or truncate inputs. A different execution plan requires a new name and reviewed profiling; preserve old evidence.

In [ ]:
assert not CPU_REPORT_ONLY
assert INPUTS_REVIEWED and INPUT_REVIEW_NOTES.strip(), 'Input review is required.'
assert BENCHMARK_REVIEWED and BENCHMARK_NOTES.strip(), 'Review section 9 before freezing.'
BATCH_SIZE = SELECTED_BATCH_SIZE or PROFILE['provisional_fastest_batch']
assert any(row['batch_size'] == BATCH_SIZE and row['status'] == 'passed' for row in PROFILE['measurements'])
run_command(['activation-freeze', '--config', PREPARED_PATH, '--profile', PROFILE_DIR,
             '--batch-size', BATCH_SIZE, '--name', RUN_NAME,
             '--review-notes', 'Input review: ' + INPUT_REVIEW_NOTES + '\nProfile review: ' + BENCHMARK_NOTES], 'freeze')
EXECUTION, PREPARED, ROWS = restore_execution()
MEASUREMENT_DIR = ARTIFACT_ROOT / EXECUTION['run_path']
print('Frozen execution:', EXECUTION_PATH)
print('Batch:', EXECUTION['batch_size'], '| examples:', len(ROWS), '| batches:', len(EXECUTION['batches']))
print('Main batch/checkpoint combinations:', len(EXECUTION['batches']) * 5)
print('Measurement artifacts:', MEASUREMENT_DIR)

In [ ]:
assert not CPU_REPORT_ONLY
EXECUTION, _, _ = restore_execution()
MEASUREMENT_DIR = ARTIFACT_ROOT / EXECUTION['run_path']
run_command(['activation-measure', '--config', EXECUTION_PATH], 'measure', MEASUREMENT_DIR)
print('Next: verify scientific completion in section 11 before disconnecting the GPU.')

## 11. Verify scientific completion — then disconnect the GPU

This is a CPU-only integrity check: all frozen combinations, payload hashes, shapes, masks, hooks, base identity, rank-1 evidence and reconstructed aggregate sums must agree. A successful process exit or a status saying completed is insufficient. Partial evidence produces progress/diagnostics, not a scientific report.

**After this cell passes and no child is active, you can disconnect/delete the GPU runtime.** Everything required for reporting is in Drive. To continue in a CPU runtime, rerun 1, install CPU reporting dependencies if necessary/restart, then 3, 4, 11, 12, 13 with `CPU_REPORT_ONLY=True`. Do not relocate or remove the source adapter files: reporting verifies their identities too.

In [ ]:
EXECUTION, _, _ = restore_execution()
MEASUREMENT_DIR = ARTIFACT_ROOT / EXECUTION['run_path']
run_command(['activation-verify', '--config', EXECUTION_PATH], 'verify', MEASUREMENT_DIR)
from pilot_eval.activation_measurement import verify_measurement
COMPLETION = verify_measurement(EXECUTION_PATH, ARTIFACT_ROOT)
assert COMPLETION['measurement_complete'] and COMPLETION['all_gates_passed']
assert not live_children(), 'Inspect active children before disconnecting.'
print('Verified examples:', COMPLETION['example_count'], '| corpus counts:', COMPLETION['corpus_counts'])
print('Completed combinations:', COMPLETION['completed_combinations'])
print('All measurement artifacts verified. GPU is no longer needed for sections 12–13.')

## 12. Reconstruct the CPU report

No model inference is performed. The shared reporting command reconstructs both weightings and primary/secondary measurements from verified sufficient summaries. It resamples **examples 2,000 times, seed 42**: GSM8K draws are paired across question/solution views, layers, projections and checkpoints; FineWeb draws are independent. Nonlinear ratios are recomputed per draw.

Exactly zero denominators produce null measurements. Undefined replicates are reported with coverage, not converted to observed zeros. Small nonzero denominators stay defined without an epsilon, with magnitude diagnostics for inspection. Intervals condition on these cohorts and one training seed.

Expected: structured results, mean vectors, exact bootstrap draws/defined masks, tables, validation diagnostics and PNG/SVG plots. An interrupted unmarked report can be reconstructed without GPU work; complete reports are verified and reused. A corrupt marked report stops rather than being silently overwritten.

In [ ]:
run_command(['activation-report', '--config', EXECUTION_PATH, '--name', REPORT_NAME], 'report', REPORT_DIR)
print('Report directory:', REPORT_DIR)

## 13. Read the figures, coverage and provenance

Block curves have token-view rows, weighting columns and checkpoint lines with 95% intervals. Module heatmaps have 28 layer rows and seven projection columns, separated by checkpoint/view; both weighting figures share a scale. Gray cells mean undefined. The denominator plot contrasts mean baseline magnitude with magnitude of the mean baseline vector: direction cancellation can make them differ.

Read the validation table and undefined/zero coverage beside the figures. The below-resolution fraction refers to sampled validation coordinates, not every measured token.

**Pinned question:** do longer gold solutions dominate token-weighted write profiles, and does equal-example weighting change them? Fixed gold lengths are not generated lengths. Differences between FineWeb and GSM8K also reflect context and length differences; these plots do not establish a causal explanation, representative web behavior or where to place LoRA.

Artifacts remain in `SPAR/spar-pilot/artifacts/reports/<REPORT_NAME>/`. `results.json` records all scalar measurements and source provenance; `mean-vectors.npz` holds directions for future analysis, not a steering experiment. Bootstrap values have separate defined masks: masked zeros are storage placeholders, not measurements.

In [ ]:
from IPython.display import Markdown, Image, display
RESULTS = json.loads((REPORT_DIR / 'results/results.json').read_text())
display(Markdown((REPORT_DIR / 'results/report.md').read_text()))
for filename in ['block-depth.png', 'module-heatmaps-token.png', 'module-heatmaps-example.png', 'block-denominators.png']:
    display(Image(filename=str(REPORT_DIR / 'plots' / filename)))
print('Structured measurements:', REPORT_DIR / 'results/results.json')
print('Scalar tables:', REPORT_DIR / 'results/measurements.csv')
print('Mean vectors:', REPORT_DIR / 'results/mean-vectors.npz')
print('Sampled validation:', REPORT_DIR / 'results/validation.jsonl')
print('Undefined primary cells:', sum(row['undefined_write'] for row in RESULTS['measurements']))
print('Source execution:', RESULTS['source_run'])
print('Source checkpoints:', json.dumps(RESULTS['provenance']['prepared']['source_evidence']['checkpoints'], indent=2))

## 14. Recovery reference — inspect before rerunning

- **Cell interrupted, child still active:** inspect the saved process record/PID and console tail below. Do not start another writer. A notebook interruption does not guarantee the CLI child stopped. If `ACTIVE_PROCESS` survives in this kernel, its `poll()` is authoritative; otherwise inspect `/proc/<pid>/cmdline` before treating a PID as the original child.
- **Missing/empty/partial progress JSON:** the monitor retains a labelled last-known snapshot. Wait for the active process; final scientific payload verification remains strict.
- **Runtime lost:** rerun 1, 3, 4. For measurement restore the same execution and rerun its stage; completed shards are verified and reused. Hardware/software/code changes can invalidate runtime identity, so do not assume a new GPU can resume the old plan.
- **Production OOM:** preserve logs/shards. Inspect capacity; a different batch/runtime needs a newly profiled/reviewed named execution. Changing a form value cannot rewrite a frozen plan. Do not alter numerical thresholds or input lengths.
- **Stale code after updating Git:** stop or inspect active children first. Changing files does not replace imported Python modules. Restart the session, run setup/environment/helpers again; every new CLI process imports the pinned checkout fresh. Preserve local edits before checkout.
- **Corrupt marked payload or failed identity gate:** stop and inspect the indicated evidence. Do not delete completion markers, invent missing source metadata or report a partial run as complete.
- **Wrong folder:** Drive contains durable artifacts; `/content/spar-pilot` is only the code checkout. Keep existing run names and internal files unchanged. The notebook/report print actual artifact paths.

No recovery action below deletes files, kills a process or changes the execution policy.

In [ ]:
print('Process records:', SESSION_DIR)
for path in sorted(SESSION_DIR.glob('*.process.json')):
    try:
        record = json.loads(path.read_text())
        print(path.name, record)
        cmdline = Path('/proc') / str(record['pid']) / 'cmdline'
        if cmdline.exists():
            print('Current PID command:', cmdline.read_bytes().replace(b'\0', b' ').decode(errors='replace'))
    except (OSError, ValueError) as error:
        print('Record unavailable:', path, error)
if 'ACTIVE_PROCESS' in globals():
    print('Last child return code (None = still active):', ACTIVE_PROCESS.poll())
for path in sorted(SESSION_DIR.glob('*.console.log')):
    print('\n', path.name, '\n', '\n'.join(path.read_text().splitlines()[-8:]))